# OptiCell Stage 2 — CTC **TRA / DET / LNK** scoring (Colab)

Measures tracking accuracy against Cell Tracking Challenge ground truth using [`traccuracy`](https://github.com/live-image-tracking-tools/traccuracy).

**Pipeline per sequence**
1. Download CTC training zip (images + `01_GT/TRA`)
2. Run OptiCell Stage-2 (`threshold` + tracking) **or** reuse prior outputs
3. Export to CTC RES (`maskXXX.tif` + `res_track.txt`)
4. Score with `CTCMatcher` + `CTCMetrics` → **TRA, DET, LNK** (measured only)

| Dataset | Seq | Notes |
|---------|-----|-------|
| Fluo-N2DH-GOWT1 | 01, 02 | Start here (~53 MB) |
| Fluo-N2DH-SIM+ | 01, 02 | Simulated nuclei |
| Fluo-N2DL-HeLa | 01, 02 | Denser |

Cite CTC Nature Methods papers if you publish. No invented scores.

## 0. Setup

In [ ]:
import os, sys, json, zipfile, urllib.request, shutil, subprocess, re
from pathlib import Path

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/opticell_tra')
else:
    BASE = Path('/content/opticell_tra')
BASE.mkdir(parents=True, exist_ok=True)
print('BASE', BASE)

In [ ]:
REPO = Path('/content/Virelion-OptiCell')
if not REPO.exists():
    !git clone https://github.com/Virelion-Biotech/Virelion-OptiCell.git
else:
    !git -C /content/Virelion-OptiCell fetch origin main
    !git -C /content/Virelion-OptiCell reset --hard origin/main

%cd /content/Virelion-OptiCell
!pip install -e . -q
!pip install -q traccuracy tifffile

from traccuracy import run_metrics
from traccuracy.loaders import load_ctc_data
from traccuracy.matchers import CTCMatcher
from traccuracy.metrics import CTCMetrics
print('traccuracy OK, git', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())

## 1. Download CTC + helpers

In [ ]:
DATASETS = {
    'Fluo-N2DH-GOWT1': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip',
        'mb': 53,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DH-SIM+': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-SIM+.zip',
        'mb': 91,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DL-HeLa': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip',
        'mb': 182,
        'sequences': ['01', '02'],
    },
}

CTC_ROOT = BASE / 'ctc'
CTC_ROOT.mkdir(parents=True, exist_ok=True)

def download_and_extract(name: str) -> Path:
    meta = DATASETS[name]
    dest = CTC_ROOT / name
    zpath = CTC_ROOT / f'{name}.zip'
    if dest.exists() and any(dest.iterdir()):
        print('[skip]', dest)
        return dest
    print(f'[download] {name} (~{meta["mb"]} MB)')
    urllib.request.urlretrieve(meta['url'], zpath)
    with zipfile.ZipFile(zpath, 'r') as zf:
        zf.extractall(CTC_ROOT)
    if not dest.exists():
        raise FileNotFoundError(dest)
    return dest

def find_seq_and_gt(root: Path, sequence: str):
    seq = root / sequence
    gt_tra = root / f'{sequence}_GT' / 'TRA'
    if not seq.is_dir():
        raise FileNotFoundError(seq)
    if not gt_tra.is_dir():
        raise FileNotFoundError(gt_tra)
    man = gt_tra / 'man_track.txt'
    if not man.is_file():
        raise FileNotFoundError(man)
    return seq, gt_tra, man

print(list(DATASETS))

In [ ]:
sys.path.insert(0, str(REPO / 'scripts'))
from export_ctc_res import export_ctc_res  # noqa: E402

def run_stage2(seq_dir: Path, out_dir: Path, max_frames: int = 0,
               track_max_distance: float = 50.0, track_max_gap: int = 1):
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    cmd = [
        sys.executable, str(REPO / 'scripts' / 'run_killer_workflow.py'),
        str(seq_dir), '-o', str(out_dir),
        '--backend', 'threshold', '--enable-tracking',
        '--track-max-distance', str(track_max_distance),
        '--track-max-gap', str(track_max_gap),
    ]
    if max_frames > 0:
        cmd += ['--max-images', str(max_frames)]
    print(' ', ' '.join(cmd[-8:]))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:] if r.stdout else '')
        print(r.stderr[-2000:] if r.stderr else '')
        raise RuntimeError(f'stage2 failed {r.returncode}')
    if not (out_dir / 'tracks.csv').is_file():
        raise RuntimeError('tracks.csv missing')
    return out_dir

def score_tra(gt_tra: Path, man_track: Path, res_dir: Path, name: str):
    gt = load_ctc_data(str(gt_tra), str(man_track), name=f'{name}_GT')
    pred = load_ctc_data(str(res_dir), str(res_dir / 'res_track.txt'), name=f'{name}_RES')
    results, _matched = run_metrics(
        gt_data=gt,
        pred_data=pred,
        matcher=CTCMatcher(),
        metrics=[CTCMetrics()],
    )
    # results is a list of metric result dicts
    block = results[0] if isinstance(results, list) else results
    if hasattr(block, 'results'):
        metrics = dict(block.results)
    elif isinstance(block, dict) and 'results' in block:
        metrics = dict(block['results'])
    else:
        metrics = dict(block)
    return metrics

def run_one(dataset: str, sequence: str, max_frames: int = 0,
            track_max_distance: float = 50.0):
    root = download_and_extract(dataset)
    seq_dir, gt_tra, man = find_seq_and_gt(root, sequence)
    tag = f'{dataset}_{sequence}'
    stage2_dir = BASE / 'stage2' / tag
    res_dir = BASE / 'res' / tag

    print(f'=== {tag} Stage-2 ===')
    run_stage2(seq_dir, stage2_dir, max_frames=max_frames,
               track_max_distance=track_max_distance)

    print(f'=== {tag} export CTC RES ===')
    if res_dir.exists():
        shutil.rmtree(res_dir)
    export_ctc_res(stage2_dir, res_dir)

    print(f'=== {tag} TRA ===')
    metrics = score_tra(gt_tra, man, res_dir, tag)
    row = {
        'dataset': dataset,
        'sequence': sequence,
        'backend': 'threshold',
        'track_max_distance': track_max_distance,
        'TRA': metrics.get('TRA'),
        'DET': metrics.get('DET'),
        'LNK': metrics.get('LNK'),
        'AOGM': metrics.get('AOGM'),
        'fn_nodes': metrics.get('fn_nodes'),
        'fp_nodes': metrics.get('fp_nodes'),
        'fn_edges': metrics.get('fn_edges'),
        'fp_edges': metrics.get('fp_edges'),
        'ns_nodes': metrics.get('ns_nodes'),
        'ws_edges': metrics.get('ws_edges'),
    }
    print(json.dumps(row, indent=2))
    out_json = BASE / 'scores' / f'{tag}_tra.json'
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps({'summary': row, 'raw': metrics}, indent=2, default=str))
    return row

print('helpers ready')

## 2. Score one-by-one

`MAX_FRAMES = 0` = full sequence. Use `20` for a quick TRA smoke test.

**Note:** TRA uses **all** GT frames. If you set `MAX_FRAMES > 0`, scores are only meaningful if you also restrict GT (not done here) — prefer `0` for publishable TRA.

In [ ]:
MAX_FRAMES = 0
TRACK_DIST = 50.0
rows = []

# --- 1/6 GOWT1 01 ---
rows.append(run_one('Fluo-N2DH-GOWT1', '01', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
# --- 2/6 GOWT1 02 ---
rows.append(run_one('Fluo-N2DH-GOWT1', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
# --- 3/6 SIM+ 01 ---
rows.append(run_one('Fluo-N2DH-SIM+', '01', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
# --- 4/6 SIM+ 02 ---
rows.append(run_one('Fluo-N2DH-SIM+', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
# --- 5/6 HeLa 01 ---
rows.append(run_one('Fluo-N2DL-HeLa', '01', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

In [ ]:
# --- 6/6 HeLa 02 ---
rows.append(run_one('Fluo-N2DL-HeLa', '02', max_frames=MAX_FRAMES, track_max_distance=TRACK_DIST))

## 3. Aggregate measured TRA table

In [ ]:
import pandas as pd

# reload from disk if session restarted mid-way
disk_rows = []
for p in sorted((BASE / 'scores').glob('*_tra.json')):
    disk_rows.append(json.loads(p.read_text())['summary'])
table = pd.DataFrame(disk_rows if disk_rows else rows)
display(table)
out_csv = BASE / 'scores' / 'tra_aggregate.csv'
table.to_csv(out_csv, index=False)
print('Wrote', out_csv)
print('Paste TRA/DET/LNK columns back for the Stage-2 TRA report — measured only.')

### Notes

- **TRA / DET / LNK** are official-style CTC metrics via `traccuracy.CTCMetrics`.
- OptiCell linker does **not** model divisions yet → parent IDs in `res_track.txt` are `0`. Division errors can lower TRA.
- Threshold segmentation quality bounds DET/TRA; hybrid/Cellpose may differ.
- Cite: Cell Tracking Challenge + traccuracy.